In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.manifold import TSNE
import matplotlib.cm as cm
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from sklearn.preprocessing import OneHotEncoder
import warnings
warnings.filterwarnings('ignore')

# For reproducibility
np.random.seed(42)

df_model = pd.read_csv(r"")
df = df_model[df_model["ReferenceDate"] == ""]

# PART 1: DATA EXPLORATION AND PREPARATION

# 1. Feature Selection for Segmentation
segmentation_features = [
    # Demographic features
    'University_enc', 'Gender_enc', 'AgeWithWayay',
    
    # Balance and spending behavior
    'last_3m_avg_balance', 'last_3m_avg_d_spend', 'last_3m_avg_p_spend',
    'spend_to_balance_ratio',
    
    # Activity metrics
    'last_3m_avg_logins', 'months_since_last_deposit',
    
    # Behavior flags
    'had_debit_last_month', 'had_prepaid_last_month', 'had_login_last_month',
    'high_debit_spender_flag'
]

X = df[segmentation_features].copy()

X.fillna(X.mean(), inplace=True)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled_df = pd.DataFrame(X_scaled, columns=segmentation_features)

# PART 2: DETERMINING OPTIMAL NUMBER OF CLUSTERS
inertia = []
silhouette_scores = []
k_range = range(2, 11)  # Testing from 2 to 10 clusters

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    inertia.append(kmeans.inertia_)
    silhouette_scores.append(silhouette_score(X_scaled, kmeans.labels_))

# Plotting Elbow Method
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(k_range, inertia, 'o-', linewidth=2, markersize=8)
plt.grid(True)
plt.xlabel('Number of Clusters (k)', fontsize=12)
plt.ylabel('Inertia', fontsize=12)
plt.title('Elbow Method for Optimal k', fontsize=14)

plt.subplot(1, 2, 2)
plt.plot(k_range, silhouette_scores, 'o-', linewidth=2, markersize=8)
plt.grid(True)
plt.xlabel('Number of Clusters (k)', fontsize=12)
plt.ylabel('Silhouette Score', fontsize=12)
plt.title('Silhouette Score Method', fontsize=14)

plt.tight_layout()
plt.savefig('optimal_k_methods.png')
plt.close()



In [ ]:
optimal_k = 5  # This would usually be determined by analyzing the elbow plot or silhouette scores

# PART 3: APPLYING K-MEANS CLUSTERING
kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
df['cluster'] = kmeans.fit_predict(X_scaled)

# Extract cluster centers and convert to original scale
cluster_centers = pd.DataFrame(
    scaler.inverse_transform(kmeans.cluster_centers_),
    columns=segmentation_features
)

# PART 4: VISUALIZING CLUSTERS

# 1. Dimensionality Reduction for Visualization
# Using PCA to reduce to 2 dimensions for visualization
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

# Add PCA components to the dataframe for visualization
df['pca_1'] = X_pca[:, 0]
df['pca_2'] = X_pca[:, 1]

# Visualize clusters in PCA space
plt.figure(figsize=(12, 10))
scatter = plt.scatter(df['pca_1'], df['pca_2'], c=df['cluster'], cmap='viridis', 
                     alpha=0.7, s=60, edgecolors='k', linewidths=0.5)
centers = pca.transform(kmeans.cluster_centers_)
plt.scatter(centers[:, 0], centers[:, 1], c='red', s=200, alpha=0.8, marker='X')

plt.colorbar(scatter, label='Cluster')
plt.xlabel('Principal Component 1', fontsize=12)
plt.ylabel('Principal Component 2', fontsize=12)
plt.title('Customer Segments - PCA Visualization', fontsize=14)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('pca_clusters.png')
plt.close()

# 2. Using t-SNE for better visualization of high-dimensional data
tsne = TSNE(n_components=2, perplexity=30, random_state=42)
X_tsne = tsne.fit_transform(X_scaled)

# Add t-SNE components to dataframe
df['tsne_1'] = X_tsne[:, 0]
df['tsne_2'] = X_tsne[:, 1]

# Visualize clusters in t-SNE space
plt.figure(figsize=(12, 10))
scatter = plt.scatter(df['tsne_1'], df['tsne_2'], c=df['cluster'], cmap='viridis', 
                     alpha=0.7, s=60, edgecolors='k', linewidths=0.5)
plt.colorbar(scatter, label='Cluster')
plt.xlabel('t-SNE Feature 1', fontsize=12)
plt.ylabel('t-SNE Feature 2', fontsize=12)
plt.title('Customer Segments - t-SNE Visualization', fontsize=14)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('tsne_clusters.png')
plt.close()